In [1]:
import sys
sys.path.append('../gpt')

import torch
import torch.nn as nn
from torch.nn import functional as F
import os
import re
from gpt_v1 import GPTLanguageModel


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device
max_iters = 6000
learning_rate = 3e-4
block_size = 8
batch_size = 4
eval_iters = 500
n_emb = 384
n_layers = 4
n_head = 4

In [3]:
with open("../data/poem.txt") as txt:
    texts = txt.read()

In [4]:
def separate_punctuation(text):
    # Use regular expression to separate commas, single quotes, and '\n'
    separated_text = re.sub(r"([',.;\n])", r" \1 ", text)
    separated_text = separated_text.replace('\n', '<n>')
    return separated_text

In [5]:
separated_text = separate_punctuation(texts)

In [6]:
words = separated_text.split()

# Count the total number of words
vocab_size = len(words)

print(f"Total number of words: {vocab_size}")

Total number of words: 23270


In [7]:
word2id = {ch:i for i, ch in enumerate(words)}
id2word = {i:ch for i, ch in enumerate(words)}
encoder = lambda s: [word2id[w] for w in s.split()]
decoder = lambda l: ' '.join([id2word[i] for i in l])

In [8]:
encoded_data = encoder(separate_punctuation(texts))
data = torch.tensor(encoded_data, dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([23270]) torch.int64
tensor([  113, 23252, 23036, 22308, 23263, 23267, 23258,     7, 21686, 23262,
        23263, 22066, 18740, 23263, 21752,    15, 20123, 23262, 23263,   113,
        23252, 23036, 22308, 23267, 23258,    25, 23252, 23263, 20515, 21816,
        23262, 23263,   113, 23252,    34, 23262, 23263, 23203,    38, 23252,
        23263,    41, 22227, 22475, 23233,  9364, 23262, 23263,   113, 23252,
        23036, 22308, 23267, 22378, 16646, 23262, 23263, 23203, 17698, 22431,
        23206,    61, 22811, 23252, 23263,    65, 22227, 23206, 22041, 23252,
        23263, 17824,    72, 23267,    74, 23262, 23263,   113, 23252, 23267,
        23258,  6670, 23263, 23260,    84, 19774,  7850, 23252, 23263, 23260,
           90,    91, 23252, 23263, 23260,    95, 23233,    97, 23262, 23263])


In [9]:
n = int(0.9*len(separate_punctuation(texts).split()))
train_data = data[:n]
val_data = data[n:]

In [10]:
train_data[:block_size+1]

tensor([  113, 23252, 23036, 22308, 23263, 23267, 23258,     7, 21686])

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target: {target}")

when input is tensor([113]) the target: 23252
when input is tensor([  113, 23252]) the target: 23036
when input is tensor([  113, 23252, 23036]) the target: 22308
when input is tensor([  113, 23252, 23036, 22308]) the target: 23263
when input is tensor([  113, 23252, 23036, 22308, 23263]) the target: 23267
when input is tensor([  113, 23252, 23036, 22308, 23263, 23267]) the target: 23258
when input is tensor([  113, 23252, 23036, 22308, 23263, 23267, 23258]) the target: 7
when input is tensor([  113, 23252, 23036, 22308, 23263, 23267, 23258,     7]) the target: 21686


In [12]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print("Inputs: ")
print(xb)
print("Targets: ")
print(yb)

Inputs: 
tensor([[ 5555, 23267, 12869, 23262, 23263, 22066, 20941, 23260],
        [23252, 22010, 22378, 21593, 23263, 22129, 22378, 18644],
        [19231, 23252, 23266, 23206,  3571, 23263, 22210, 21874],
        [23143, 18846, 23263, 21684, 22227, 23267, 18851, 23240]])
Targets: 
tensor([[23267, 12869, 23262, 23263, 22066, 20941, 23260, 23206],
        [22010, 22378, 21593, 23263, 22129, 22378, 18644, 15705],
        [23252, 23266, 23206,  3571, 23263, 22210, 21874, 20865],
        [18846, 23263, 21684, 22227, 23267, 18851, 23240, 22475]])


In [13]:
model = GPTLanguageModel(vocab_size=vocab_size)
model.to(device)

GPTLanguageModel(
  (token_embeding_table): Embedding(23270, 384)
  (position_embedding_table): Embedding(23270, 384)
  (blocks): Sequential(
    (0): Block(
      (selfAttention): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=384, out_features=96, bias=False)
            (query): Linear(in_features=384, out_features=96, bias=False)
            (value): Linear(in_features=384, out_features=96, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
      (ln1): LayerNorm((384,), eps=1e-05, ele

In [14]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            X, Y = X.to(device), Y.to(device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [15]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"steps: {iter} train loss: {losses['train']} val loss: {losses['val']}")
        
    xb, yb = get_batch('train')
    xb = xb.to(device)
    yb = yb.to(device)

    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

steps: 0 train loss: 10.133734703063965 val loss: 10.115365028381348
steps: 500 train loss: 5.553101539611816 val loss: 6.825199604034424
steps: 1000 train loss: 4.665164947509766 val loss: 6.790243625640869
steps: 1500 train loss: 4.147660255432129 val loss: 7.120448112487793
steps: 2000 train loss: 3.610365390777588 val loss: 7.305067539215088
steps: 2500 train loss: 3.18558931350708 val loss: 7.417867660522461
steps: 3000 train loss: 2.8530611991882324 val loss: 7.4805402755737305
steps: 3500 train loss: 2.518798351287842 val loss: 7.685590744018555
steps: 4000 train loss: 2.251796245574951 val loss: 7.679314136505127
steps: 4500 train loss: 1.9435033798217773 val loss: 8.116684913635254
steps: 5000 train loss: 1.7747702598571777 val loss: 8.144585609436035
steps: 5500 train loss: 1.587455153465271 val loss: 8.306066513061523
2.046248197555542


In [17]:
model_path = 'E:\\github_clone\\project-koperundevi\\models\\trained_model.pth'
torch.save(model.state_dict(), model_path)

In [18]:
file_size = os.path.getsize(model_path)
print(f"The size of the saved model is: {file_size / 1e6:.2f} MB")

The size of the saved model is: 135.74 MB


In [22]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")


trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")


non_trainable_params = total_params - trainable_params
print(f"Non-trainable parameters: {non_trainable_params}")

Total parameters: 33924326
Trainable parameters: 33924326
Non-trainable parameters: 0


In [ ]:
33,924,326